# 03 — Huấn luyện, Đánh giá và Trích xuất Embedding trên Yoochoose: Mô hình GRU4Rec

Notebook này chạy được trực tiếp trên cả **Kaggle GPU (T4 / P100)** lẫn **máy cục bộ (Local)** để huấn luyện và đánh giá mô hình **GRU4Rec** (*Hidasi et al., ICLR 2016*) trên bộ dữ liệu **Yoochoose `1/32`** với **2 random seed (`2024`, `2025`)**.

- **Kiến trúc**: Item Embedding → Dropout → GRU → Dense Projection → Full-Catalog Dot Product
- **Cấu hình**: `configs/yoochoose/common.yaml` + `configs/yoochoose/data.yaml` + `configs/gru4rec.yaml` (dùng chung kiến trúc với Diginetica để so sánh công bằng)

### Cách chạy gọn trên Kaggle GPU (đầu ra `/kaggle/working` siêu sạch)
1. Upload file `yoochoose_kaggle_input.zip` thành 1 **Kaggle Dataset** rồi gắn (`Add Input`) vào notebook.
2. Bật **Settings → Accelerator → GPU T4 x2** (hoặc **GPU P100**).
3. Nhấn **Run All**:
   - **Bước 0** tự copy input từ `/kaggle/input/...` sang không gian làm việc tạm **`/tmp/da3_work`** (KHÔNG xả rác vào `/kaggle/working`).
   - **Bước 8** tự gom đúng các file đầu ra và đóng gói thành **DUY NHẤT 1 file** `/kaggle/working/yoochoose_gru4rec_outputs.zip` giữ nguyên đúng đường dẫn tương đối (`runs/yoochoose/gru4rec/...`, `results/yoochoose/item_level/...`, `results/yoochoose/tables/training_runtime_gru4rec.csv`).
4. Tải file `yoochoose_gru4rec_outputs.zip` về và **giải nén thẳng vào `D:\DA-3\`** — mọi file tự khớp đúng vị trí, không cần lọc thủ công và không ghi đè mô hình khác.

## 0. Thiết lập môi trường Kaggle GPU / Local (cô lập không gian tạm `/tmp/da3_work` để giữ sạch `/kaggle/working`)

In [ ]:
from __future__ import annotations

import os
import shutil
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()
KAGGLE_WORKING_DIR = Path("/kaggle/working")
KAGGLE_TMP_WORK = Path("/tmp/da3_work")

if IS_KAGGLE:
    print("[KAGGLE MODE] Phát hiện môi trường Kaggle.")
    matches = list(Path("/kaggle/input").rglob("BTL-K03-tong-quan-de-tai.md"))
    if not matches:
        raise FileNotFoundError(
            "Không tìm thấy 'BTL-K03-tong-quan-de-tai.md' trong /kaggle/input! "
            "Hãy Add Input dataset chứa file yoochoose_kaggle_input.zip."
        )
    source_root = matches[0].parent
    print(f"Nguồn input tìm thấy tại : {source_root}")

    if KAGGLE_TMP_WORK.exists():
        shutil.rmtree(KAGGLE_TMP_WORK)
    KAGGLE_TMP_WORK.mkdir(parents=True, exist_ok=True)

    # Copy input sang /tmp/da3_work (nhanh, có quyền ghi, giữ /kaggle/working sạch hoàn toàn)
    for rel in ("BTL-K03-tong-quan-de-tai.md", "configs", "src", "data"):
        src_item = source_root / rel
        if not src_item.exists():
            raise FileNotFoundError(f"Thiếu thành phần bắt buộc trong Kaggle Dataset: {rel}")
        dst_item = KAGGLE_TMP_WORK / rel
        if src_item.is_dir():
            shutil.copytree(src_item, dst_item, dirs_exist_ok=True)
        else:
            shutil.copy2(src_item, dst_item)

    os.chdir(str(KAGGLE_TMP_WORK))
    print(f"Thư mục làm việc tạm     : {Path.cwd()}")
    print(f"Thư mục xuất zip sạch    : {KAGGLE_WORKING_DIR}")
else:
    print(f"[LOCAL MODE] Chạy trực tiếp trên máy cục bộ: {Path.cwd()}")

## 1. Thiết lập môi trường, thiết bị (GPU/CPU) và đọc cấu hình Yoochoose

In [ ]:
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "BTL-K03-tong-quan-de-tai.md").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc dự án BTL-K03.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from btl_k03.paths import get_dataset_paths, ensure_output_dirs
from btl_k03.utils import load_experiment_config, load_item_mapping, set_global_seed
from btl_k03.models import SessionPrefixDataset, build_model_from_config, train_and_evaluate_seed

DATASET = "yoochoose"
MODEL_NAME = "gru4rec"

ensure_output_dirs(DATASET)
YOO_PATHS = get_dataset_paths(DATASET)
PROCESSED_YOO_DIR = YOO_PATHS["processed_dir"]
CONFIG = load_experiment_config(MODEL_NAME, dataset=DATASET)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if IS_KAGGLE and not torch.cuda.is_available():
    print("CẢNH BÁO: Bạn đang chạy trên Kaggle nhưng CHƯA BẬT GPU! Hãy vào Settings -> Accelerator -> chọn GPU T4 x2 hoặc P100.")

gpu_info = (
    f"{torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)"
    if torch.cuda.is_available()
    else "CPU"
)
print(f"PROJECT_ROOT      : {PROJECT_ROOT}")
print(f"Python executable : {sys.executable}")
print(f"PyTorch version   : {torch.__version__}")
print(f"Device            : {DEVICE} ({gpu_info})")
print(f"Dataset / Model   : {DATASET.upper()} / {CONFIG['architecture']}")
print(f"Seeds             : {CONFIG['seeds']}")
print(f"Model params      : {json.dumps(CONFIG['model_params'], ensure_ascii=False)}")
print(f"Training config   : {json.dumps(CONFIG['training'], ensure_ascii=False)}")

## 2. Kiểm tra dữ liệu đầu vào Yoochoose và mặt nạ Warm / Controlled Cold (`Option B`)

In [ ]:
item_mapping = load_item_mapping(PROCESSED_YOO_DIR)
train_examples = pd.read_parquet(PROCESSED_YOO_DIR / "train_examples.parquet")
val_examples = pd.read_parquet(PROCESSED_YOO_DIR / "validation_examples.parquet")
test_examples = pd.read_parquet(PROCESSED_YOO_DIR / "test_examples.parquet")
test_last_examples = pd.read_parquet(PROCESSED_YOO_DIR / "test_last_item_examples.parquet")

n_total_catalog = len(item_mapping)
n_warm = int(item_mapping["is_trainable_warm"].sum())
n_cold = int(item_mapping["is_controlled_cold"].sum())
n_reg_cat = int(item_mapping["has_regular_category"].sum()) if "has_regular_category" in item_mapping.columns else 0
n_special = int(item_mapping["ever_special_offer"].sum()) if "ever_special_offer" in item_mapping.columns else 0

summary_input = pd.DataFrame([
    {"Chỉ tiêu": "Tổng số item trong bảng Embedding (ID 1..N)", "Giá trị": f"{n_total_catalog:,}"},
    {"Chỉ tiêu": "Số item Warm hợp lệ (is_trainable_warm = True)", "Giá trị": f"{n_warm:,}"},
    {"Chỉ tiêu": "Số item Controlled Cold 20% (is_controlled_cold = True)", "Giá trị": f"{n_cold:,}"},
    {"Chỉ tiêu": "Số item có Category chuẩn 1..12", "Giá trị": f"{n_reg_cat:,} ({n_reg_cat / n_total_catalog * 100:.2f}%)"},
    {"Chỉ tiêu": "Số item từng gắn nhãn S (Special Offer)", "Giá trị": f"{n_special:,} ({n_special / n_total_catalog * 100:.2f}%)"},
    {"Chỉ tiêu": "Số mẫu Train (All-prefix)", "Giá trị": f"{len(train_examples):,}"},
    {"Chỉ tiêu": "Số mẫu Validation Warm (All-prefix)", "Giá trị": f"{len(val_examples):,}"},
    {"Chỉ tiêu": "Số mẫu Test Warm chính (All-prefix)", "Giá trị": f"{len(test_examples):,}"},
    {"Chỉ tiêu": "Số mẫu Test Warm phiên cuối (Last-item)", "Giá trị": f"{len(test_last_examples):,}"},
])
display(summary_input)

## 3. Khởi tạo kiến trúc GRU4Rec và Sanity Check khóa Gradient Cold Item

Chạy kiểm tra 1 batch giả lập (64 mẫu) trước khi huấn luyện chính thức để xác nhận:
1. `logits` của `padding_idx = 0` và toàn bộ Controlled Cold items bị che bằng `-1e9`, loss khởi tạo xấp xỉ $\ln(N_{\text{warm}}) \approx 9{,}03$.
2. Sau `loss.backward()`, đạo hàm (`grad`) của hàng padding và toàn bộ Controlled Cold items bằng `0.0` tuyệt đối, trong khi các hàng Warm nhận gradient bình thường.

In [ ]:
set_global_seed(2024)
sanity_model = build_model_from_config(CONFIG, item_mapping, DEVICE)
print(sanity_model)

sample_ds = SessionPrefixDataset(train_examples.head(64), max_seq_length=CONFIG["max_seq_length"])
sample_seq = sample_ds.sequences.to(DEVICE)
sample_len = sample_ds.lengths.to(DEVICE)
sample_target = sample_ds.targets_tensor.to(DEVICE)

sanity_model.zero_grad()
logits = sanity_model.full_sort_logits(sample_seq, sample_len, mask_cold_and_pad=True)
loss = torch.nn.functional.cross_entropy(logits, sample_target)
loss.backward()

grad_matrix = sanity_model.item_embedding.weight.grad.detach().cpu().numpy()
cold_ids = item_mapping.loc[item_mapping["is_controlled_cold"], "internal_item_id"].to_numpy()
warm_ids = item_mapping.loc[item_mapping["is_trainable_warm"], "internal_item_id"].to_numpy()

max_cold_grad = float(np.max(np.abs(grad_matrix[cold_ids])))
max_pad_grad = float(np.max(np.abs(grad_matrix[0])))
max_warm_grad = float(np.max(np.abs(grad_matrix[warm_ids])))

print("\n[SANITY CHECK GRADIENT — YOOCHOOSE]")
print(f"- Loss khởi tạo (Cross-Entropy trên {n_warm:,} warm items): {loss.item():.4f} (Lý thuyết ~ ln({n_warm}) = {np.log(n_warm):.4f})")
print(f"- Max |grad| của hàng Padding (ID 0)             : {max_pad_grad:.10f}")
print(f"- Max |grad| của {n_cold:,} hàng Controlled Cold       : {max_cold_grad:.10f}")
print(f"- Max |grad| của {n_warm:,} hàng Trainable Warm        : {max_warm_grad:.10f}")
assert max_cold_grad == 0.0 and max_pad_grad == 0.0 and max_warm_grad > 0.0
del sanity_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Huấn luyện GRU4Rec trên Yoochoose với 2 Seed (`2024` và `2025`)

> **Ghi chú:** Giữ `MAX_EPOCHS_OVERRIDE = None` để chạy đầy đủ tối đa `30` epoch kèm Early Stopping (`patience = 5` theo `Validation MRR@20`). Trên Kaggle GPU T4/P100, cả 2 seed sẽ chạy xong chỉ trong vài phút.

In [ ]:
MAX_EPOCHS_OVERRIDE = None  # Đặt None để chạy full theo config (30 epochs + Early Stopping)

run_outputs = {}
for seed in CONFIG["seeds"]:
    run_outputs[seed] = train_and_evaluate_seed(
        model_name=MODEL_NAME,
        seed=seed,
        config=CONFIG,
        processed_dir=PROCESSED_YOO_DIR,
        max_epochs_override=MAX_EPOCHS_OVERRIDE,
        dataset=DATASET,
    )
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Tổng hợp chất lượng dự đoán & Thời gian train của GRU4Rec

In [ ]:
acc_rows = []
stratified_frames = []
cold_diag_rows = []

for seed, out in run_outputs.items():
    tm = out["test_metrics"]
    rt = tm.get("runtime", {})
    w_all = tm["test_warm_all_prefix"]
    w_last = tm["test_warm_last_item"]
    acc_rows.append({
        "model": MODEL_NAME.upper(),
        "seed": seed,
        "epochs_ran": rt.get("epochs_ran"),
        "best_epoch": tm["best_epoch"],
        "train_time_min": rt.get("total_train_time_min"),
        "mean_epoch_sec": rt.get("mean_epoch_time_sec"),
        "total_seed_min": rt.get("total_seed_wall_time_min"),
        "val_best_mrr@20": tm["best_validation_score"],
        "test_warm_recall@20": w_all["recall@20"],
        "test_warm_mrr@20": w_all["mrr@20"],
        "test_warm_ndcg@20": w_all["ndcg@20"],
        "test_last_recall@20": w_last["recall@20"],
        "test_last_mrr@20": w_last["mrr@20"],
        "test_last_ndcg@20": w_last["ndcg@20"],
        "cold_max_drift": tm["cold_embedding_max_abs_drift"],
    })
    strat_df = out["test_stratified_df"].copy()
    strat_df.insert(0, "seed", seed)
    stratified_frames.append(strat_df)
    cold_diag_rows.append({"seed": seed, **tm["cold_start_diagnostic"]})

print("=== BẢNG 1: CHẤT LƯỢNG DỰ ĐOÁN & THỜI GIAN TRAIN CỦA GRU4Rec TRÊN TẬP TEST WARM (YOOCHOOSE) ===")
display(pd.DataFrame(acc_rows))

per_model_rt_csv = YOO_PATHS["tables_dir"] / f"training_runtime_{MODEL_NAME}.csv"
if per_model_rt_csv.is_file():
    print(f"\n=== BẢNG THỜI GIAN TRAIN ĐÃ LƯU ({per_model_rt_csv.name}) ===")
    display(pd.read_csv(per_model_rt_csv))

print("\n=== BẢNG 2: ĐỘ CHÍNH XÁC TÁCH THEO NHÓM TARGET (POPULAR / MEDIUM / RARE) ===")
display(pd.concat(stratified_frames, ignore_index=True))

print("\n=== BẢNG 3: ĐỐI CHỨNG HIỆU NĂNG KHI TÍNH CẢ 20% CONTROLLED COLD & NATURAL COLD ===")
display(pd.DataFrame(cold_diag_rows))

## 6. Chỉ số Thiên lệch độ phổ biến ở đầu ra (`Coverage@20`, `Gini@20`, `ARP@20`) & Hiệu ứng nhãn `S`

In [ ]:
bias_rows = []
exposure_frames = []
special_offer_frames = []

for seed, out in run_outputs.items():
    pb = out["test_metrics"]["popularity_bias"]
    il = out["item_level_df"]
    warm_il = il.loc[il["is_trainable_warm"]]
    rho_fn, _ = stats.spearmanr(warm_il["train_freq"], warm_il["embedding_norm"])

    bias_rows.append({
        "model": MODEL_NAME.upper(),
        "seed": seed,
        "coverage_warm@20": pb["coverage_warm@20"],
        "coverage_full@20": pb["coverage_full@20"],
        "gini_warm@20": pb["gini_warm@20"],
        "gini_full@20": pb["gini_full@20"],
        "arp@20": pb["arp@20"],
        "arp_log1p@20": pb["arp_log1p@20"],
        "popular_exposure_pct": pb["popular_group_exposure_share_pct"],
        "rare_exposure_pct": pb["rare_group_exposure_share_pct"],
        "preview_spearman_freq_norm": float(rho_fn),
    })
    exp_df = out["group_exposure_df"].copy()
    exp_df.insert(0, "seed", seed)
    exposure_frames.append(exp_df)

    if "ever_special_offer" in warm_il.columns:
        total_slots = max(1, int(warm_il["rec_count"].sum()))
        total_train = max(1, int(warm_il["train_freq"].sum()))
        for flag, lbl in [(True, "Có nhãn S (Special Offer)"), (False, "Không nhãn S")]:
            sub = warm_il.loc[warm_il["ever_special_offer"] == flag]
            special_offer_frames.append({
                "seed": seed,
                "nhom_special_offer": lbl,
                "warm_items": len(sub),
                "warm_catalog_pct": len(sub) / len(warm_il) * 100.0,
                "train_click_share_pct": sub["train_freq"].sum() / total_train * 100.0,
                "top20_exposure_share_pct": sub["rec_count"].sum() / total_slots * 100.0,
                "mean_embedding_norm": float(sub["embedding_norm"].mean()),
            })

print("=== BẢNG 4: CHỈ SỐ POPULARITY BIAS ĐẦU RA & XEM TRƯỚC SPEARMAN(FREQ, NORM) — GRU4Rec ===")
display(pd.DataFrame(bias_rows))

print("\n=== BẢNG 5: PHÂN BỔ LƯỢT HIỂN THỊ TOP-20 THEO NHÓM POPULAR / MEDIUM / RARE / COLD ===")
display(pd.concat(exposure_frames, ignore_index=True))

if special_offer_frames:
    print("\n=== BẢNG 6 (ĐẶC THÙ YOOCHOOSE): EXPOSURE & NORM GIỮA NHÓM SPECIAL OFFER ('S') VS. KHÔNG 'S' ===")
    display(pd.DataFrame(special_offer_frames))

## 7. Biểu đồ Learning Curve, Phân bố Norm Embedding và Kiểm tra file bàn giao của GRU4Rec

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for seed, out in run_outputs.items():
    hist = out["history_df"]
    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label=f"Seed {seed}")
    axes[1].plot(hist["epoch"], hist["validation_mrr@20"], marker="s", label=f"Seed {seed} (MRR@20)")
    axes[1].plot(hist["epoch"], hist["validation_recall@20"], linestyle="--", label=f"Seed {seed} (Recall@20)")

axes[0].set_title("GRU4Rec (Yoochoose) — Training Cross-Entropy Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title("GRU4Rec (Yoochoose) — Validation Recall@20 & MRR@20")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].legend()

first_seed = CONFIG["seeds"][0]
il_df = run_outputs[first_seed]["item_level_df"]
sns.boxplot(
    data=il_df,
    x="popularity_group",
    y="embedding_norm",
    hue="popularity_group",
    order=["popular", "medium", "rare", "cold"],
    hue_order=["popular", "medium", "rare", "cold"],
    palette="Set2",
    legend=False,
    ax=axes[2],
)
axes[2].set_title(f"GRU4Rec (Yoochoose, Seed {first_seed}) — L2 Norm theo nhóm Popularity")
axes[2].set_xlabel("Nhóm Item")
axes[2].set_ylabel("L2 Norm ||e_i||_2")

plt.tight_layout()
plt.show()

REQUIRED_FILES = [
    "config.yaml",
    "checkpoint.pth",
    "training.log",
    "training_history.csv",
    "training_runtime.json",
    "test_metrics.json",
    "top20.parquet",
    "item_embeddings.npy",
    "item_embedding_mapping.csv",
]

manifest_rows = []
for seed, out in run_outputs.items():
    rdir = out["run_dir"]
    for fname in REQUIRED_FILES:
        fpath = rdir / fname
        manifest_rows.append({
            "seed": seed,
            "file": str(fpath.relative_to(PROJECT_ROOT)),
            "exists": fpath.is_file(),
            "size_kb": round(fpath.stat().st_size / 1024.0, 2) if fpath.is_file() else 0.0,
        })
    il_path = YOO_PATHS["item_level_dir"] / f"{MODEL_NAME}_seed{seed}.parquet"
    manifest_rows.append({
        "seed": seed,
        "file": str(il_path.relative_to(PROJECT_ROOT)),
        "exists": il_path.is_file(),
        "size_kb": round(il_path.stat().st_size / 1024.0, 2) if il_path.is_file() else 0.0,
    })

manifest_df = pd.DataFrame(manifest_rows)
assert manifest_df["exists"].all(), f"Thiếu file bàn giao trong runs/yoochoose/{MODEL_NAME}!"
display(manifest_df)

## 8. Đóng gói gọn đầu ra vào `/kaggle/working/yoochoose_gru4rec_outputs.zip` (chỉ chạy khi ở Kaggle)

- Thư mục `/kaggle/working/` sẽ **chỉ chứa duy nhất 1 file zip** `yoochoose_gru4rec_outputs.zip` (không có bất kỳ thư mục rác nào).
- Cấu trúc bên trong file zip giữ đúng đường dẫn gốc dự án:
  - `runs/yoochoose/gru4rec/seed_2024/...` (9 file)
  - `runs/yoochoose/gru4rec/seed_2025/...` (9 file)
  - `results/yoochoose/item_level/gru4rec_seed2024.parquet`
  - `results/yoochoose/item_level/gru4rec_seed2025.parquet`
  - `results/yoochoose/tables/training_runtime_gru4rec.csv`
- Khi tải về máy, bạn chỉ cần **giải nén thẳng vào thư mục gốc `D:\DA-3\`** — toàn bộ file tự rơi đúng thư mục và **không bao giờ ghi đè** kết quả của 2 mô hình còn lại.

In [ ]:
import zipfile

if IS_KAGGLE:
    output_zip = KAGGLE_WORKING_DIR / f"yoochoose_{MODEL_NAME}_outputs.zip"
    if output_zip.exists():
        output_zip.unlink()

    files_to_pack: list[Path] = []
    model_runs_dir = YOO_PATHS["runs_dir"] / MODEL_NAME
    for f in sorted(model_runs_dir.rglob("*")):
        if f.is_file() and not f.name.endswith(".gitkeep"):
            files_to_pack.append(f)

    for seed in CONFIG["seeds"]:
        il_file = YOO_PATHS["item_level_dir"] / f"{MODEL_NAME}_seed{seed}.parquet"
        if il_file.is_file():
            files_to_pack.append(il_file)

    per_model_rt = YOO_PATHS["tables_dir"] / f"training_runtime_{MODEL_NAME}.csv"
    if per_model_rt.is_file():
        files_to_pack.append(per_model_rt)

    with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for f in files_to_pack:
            arcname = f.relative_to(PROJECT_ROOT).as_posix()
            zf.write(f, arcname)
            print(f"  + {arcname} ({f.stat().st_size / 1024:.1f} KB)")

    print("=" * 72)
    print(f"ĐÃ ĐÓNG GÓI THÀNH CÔNG: {output_zip} ({output_zip.stat().st_size / (1024**2):.2f} MB)")
    print("Danh sách file hiện có trong /kaggle/working (siêu gọn):")
    for item in sorted(KAGGLE_WORKING_DIR.iterdir()):
        print(f"  - {item.name} ({item.stat().st_size / (1024**2):.2f} MB)" if item.is_file() else f"  - {item.name}/")
    print("Hướng dẫn: Tải file zip trên về máy và giải nén (Extract Here) thẳng vào thư mục D:\\DA-3\\")
else:
    print("[LOCAL MODE] Kết quả đã được lưu trực tiếp vào runs/yoochoose/ và results/yoochoose/, không cần đóng gói zip.")